# 23. Reviews Analysis — Sentiment Classification

**URL:** https://www.sparkplayground.com/pyspark-coding-interview-questions/reviews-analysis  
**Difficulty:** Easy  
**Date:** 2026-04-19

## Task
Classify each review into `positive` / `negative` / `neutral`:
- **positive**: `rating >= 4` AND comment contains a positive keyword (`good, excellent, love, great, amazing`)
- **negative**: `rating <= 2` AND comment contains a negative keyword (`bad, poor, hate, terrible, awful`)
- **neutral**: everything else (rating = 3, or high rating without a positive word, or low rating without a negative word)

Two solutions below:
1. **Custom UDF** — what the problem asks for.
2. **Native Spark expression** (`rlike` + `when`) — faster alternative that stays in the JVM.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

data = [
    (1,  101, 5, "Excellent product, love it!"),
    (2,  101, 2, "Poor quality, not worth it."),
    (3,  102, 3, "It's okay, nothing special."),
    (4,  103, 4, "Good value for money."),
    (5,  103, 1, "Hate the build quality."),
    (6,  104, 5, "Absolutely amazing!"),
    (7,  105, 2, "Bad experience overall."),
    (8,  106, 3, "Decent but overpriced."),
    (9,  107, 4, "Loved the packaging and speed."),
    (10, 108, 1, "Terrible. Would not recommend."),
    (11, 109, 5, "Excellent purchase. Super happy!"),
    (12, 110, 3, "Average item, nothing great."),
]
columns = ["customer_id", "product_id", "rating", "comment"]
df = spark.createDataFrame(data, columns)
df.show(truncate=False)

## Solution 1 — Custom UDF

Wraps a plain Python function with `udf(..., StringType())` so Spark can call it per row.

In [ ]:
pos_keywords = ["good", "excellent", "love", "great", "amazing"]
neg_keywords = ["bad", "poor", "hate", "terrible", "awful"]


def classify_statement(rating, comment):
    # Null guard — .lower() on None would crash the UDF for that row.
    if comment is None:
        return "neutral"

    comment = comment.lower()

    if rating >= 4 and any(word in comment for word in pos_keywords):
        return "positive"
    elif rating <= 2 and any(word in comment for word in neg_keywords):
        return "negative"
    else:
        return "neutral"


sentiment_udf = udf(classify_statement, StringType())

df_udf = (
    df.withColumn("sentiment", sentiment_udf("rating", "comment"))
      .select("customer_id", "product_id", "sentiment")
)
df_udf.show(truncate=False)

## Solution 2 — Native Spark (`rlike` + `when`)

Builds the same logic with column expressions. Runs entirely in the JVM — no Python serialization per row, so it's faster than the UDF. Prefer this whenever the logic fits built-ins.

In [ ]:
is_positive = (F.col("rating") >= 4) & (F.lower(F.col("comment")).rlike("|".join(pos_keywords)))
is_negative = (F.col("rating") <= 2) & (F.lower(F.col("comment")).rlike("|".join(neg_keywords)))

df_native = (
    df.withColumn(
        "sentiment",
        F.when(is_positive, "positive")
         .when(is_negative, "negative")
         .otherwise("neutral"),
    )
    .select("customer_id", "product_id", "sentiment")
)
df_native.show(truncate=False)

## Notes

- `any(word in comment for word in keywords)` short-circuits on the first match.
- `rlike("|".join(keywords))` does a substring match — `good` would also match inside `goods`. Fine for this dataset; use word boundaries (`\\bgood\\b`) if you need stricter matching.
- **UDF vs native:** UDFs serialize each row JVM → Python → JVM. Column expressions stay in the JVM and also allow Catalyst to optimize. Use UDFs only when the problem demands one (as here) or when the logic can't be expressed with built-ins.